# Case study: forecasting the monthly number of reviews

*Session 12 practice (all three blocks). Author: course team, licence CC-BY-4.0.*

**Question.** How many reviews will the Health and Personal Care category receive per month next year, and how certain is that forecast?

- **Part 1 (block 1):** build the monthly series, describe it and compute the baselines.
- **Part 2 (block 2):** fit exponential smoothing with a prediction interval and compare it with the baselines.
- **Part 3 (block 3):** add a lag-feature model, backtest all models with a rolling origin, and recommend one with its prediction interval.
- **Epilogue:** compare the forecast for 2022 with the 2022 reviews in the test file.

Theory: [01](../theory/01-time-series-and-baselines.md), [02](../theory/02-exponential-smoothing-and-arima.md), [03](../theory/03-lag-features-and-backtesting.md).

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
from statsmodels.tsa.seasonal import STL

warnings.filterwarnings("ignore", category=UserWarning)     # statsmodels convergence notes
pd.set_option("display.width", 120)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").exists())
DATA = ROOT / "case-study" / "data"


# Part 1: the series and its baselines

## 1.1 From reviews to a monthly series

In [ ]:
reviews = pd.read_parquet(DATA / "train.parquet", columns=["date"])
monthly_all = reviews.set_index("date").resample("MS").size()
print(monthly_all.groupby(monthly_all.index.year).sum().to_string())

**Question.** Why do we start in 2012? Look at the yearly totals above.

In [ ]:
y = monthly_all["2012":"2021"].rename("reviews")
y.index.freq = "MS"
log_y = np.log(y)
ax = y.plot(figsize=(10, 3.5), title="Reviews per month")
ax.set_ylabel("reviews")
len(y)

## 1.2 Trend, seasonality, autocorrelation

In [ ]:
stl = STL(log_y, period=12, robust=True).fit()
stl.plot()
plt.tight_layout()
seasonal_factor = np.exp(stl.seasonal).groupby(y.index.month).mean().round(2)
seasonal_factor.rename("seasonal factor").to_frame().T

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
plot_acf(y, lags=24, ax=a1, title="ACF of the counts")
plot_acf(log_y.diff().dropna(), lags=24, ax=a2, title="ACF of monthly growth")
plt.tight_layout()

**Question.** Which months are high, which low? What does the peak at lag 12 in the right-hand plot mean?

## 1.3 Baselines on the hold-out year 2021

In [ ]:
train, test = y[:"2020"], y["2021"]
H = len(test)


def mae(actual, forecast):
    return float(np.mean(np.abs(np.asarray(actual) - np.asarray(forecast))))


def mase(actual, forecast, y_tr, m=12):
    history = np.asarray(y_tr)
    return mae(actual, forecast) / np.mean(np.abs(history[m:] - history[:-m]))


def baselines(y_tr, index):
    return {
        "naive": pd.Series(y_tr.iloc[-1], index=index),
        "seasonal naive": pd.Series(y_tr.iloc[-12:].to_numpy(), index=index),
        "moving average (3)": pd.Series(y_tr.iloc[-3:].mean(), index=index),
    }


base = baselines(train, test.index)
ax = y["2018":].plot(figsize=(10, 3.5), color="black", label="actual")
for name, fc in base.items():
    fc.plot(ax=ax, ls="--", label=name)
ax.legend()
pd.DataFrame({name: {"MAE": mae(test, fc), "MASE": mase(test, fc, train)} for name, fc in base.items()}).T.round(2)

# Part 2: exponential smoothing with a prediction interval

We fit an ETS model on the log counts: additive errors, a damped additive trend and additive seasonality (on the log scale, i.e. multiplicative on the original scale).

In [ ]:
def fit_ets(y_tr):
    return ETSModel(np.log(y_tr), error="add", trend="add", damped_trend=True,
                    seasonal="add", seasonal_periods=12).fit(disp=False)


ets = fit_ets(train)
print(dict(zip(ets.param_names[:4], ets.params[:4].round(3))))
frame = np.exp(ets.get_prediction(start=test.index[0], end=test.index[-1]).summary_frame(alpha=0.05))

ax = y["2018":].plot(figsize=(10, 3.5), color="black", label="actual")
ax.fill_between(frame.index, frame["pi_lower"], frame["pi_upper"], alpha=0.2, label="95 % interval")
frame["mean"].plot(ax=ax, ls="--", label="ETS")
ax.legend(loc="upper left")
inside = ((test >= frame["pi_lower"]) & (test <= frame["pi_upper"])).sum()
print(f"ETS MAE {mae(test, frame['mean']):.0f}, MASE {mase(test, frame['mean'], train):.2f}, {inside} of {H} months inside the interval")

**Exercise 2.1.** Fit the same model without the damped trend (`damped_trend=False`) and without the log transform. Compare MAE and the width of the December interval.

In [ ]:
# Exercise 2.1: your code here


# Part 3: lag features, backtest and recommendation

## 3.1 A lag-feature model (direct, 12 months ahead)

Only lags of 12 or more months are used, so that every feature is known when the forecast is made.

In [ ]:
table = pd.DataFrame({"target": log_y})
for lag in (12, 13, 24):
    table[f"lag{lag}"] = log_y.shift(lag)
table["month"] = table.index.month
table = table.dropna()


def fit_lag_model(y_tr):
    rows = table.loc[:y_tr.index[-1]]
    return HistGradientBoostingRegressor(min_samples_leaf=5, random_state=0).fit(
        rows.drop(columns="target"), rows["target"])


lag_model = fit_lag_model(train)
lag_fc = pd.Series(np.exp(lag_model.predict(table.loc[test.index].drop(columns="target"))), index=test.index)
print(f"lag model MAE {mae(test, lag_fc):.0f}, MASE {mase(test, lag_fc, train):.2f}")

## 3.2 Rolling-origin backtest

Five folds: the origin moves forward one year at a time, the test years are 2017 to 2021.

In [ ]:
rows = []
for train_idx, test_idx in TimeSeriesSplit(n_splits=5, test_size=12).split(y):
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
    fcs = baselines(y_tr, y_te.index)
    pred = np.exp(fit_ets(y_tr).get_prediction(start=y_te.index[0], end=y_te.index[-1]).summary_frame(alpha=0.05))
    fcs["ETS"] = pred["mean"]
    fcs["lag model"] = np.exp(fit_lag_model(y_tr).predict(table.loc[y_te.index].drop(columns="target")))
    coverage = ((y_te >= pred["pi_lower"]) & (y_te <= pred["pi_upper"])).mean()
    for name, fc in fcs.items():
        rows.append({"year": y_te.index[0].year, "model": name, "MAE": mae(y_te, fc),
                     "MASE": mase(y_te, fc, y_tr), "ETS coverage": coverage})
backtest = pd.DataFrame(rows)
backtest.pivot(index="year", columns="model", values="MASE").round(2)

In [ ]:
summary = backtest.groupby("model").agg(MAE=("MAE", "mean"), MASE=("MASE", "mean"),
                                        MASE_worst=("MASE", "max"))
summary["years best"] = (backtest.loc[backtest.groupby("year")["MASE"].idxmin(), "model"]
                         .value_counts().reindex(summary.index).fillna(0).astype(int))
print("ETS interval coverage per year:", backtest.groupby("year")["ETS coverage"].first().round(2).to_list())
summary.sort_values("MASE").round(2)

## 3.3 Recommendation

Write your recommendation for the product manager (4–6 sentences): which model, its forecast for 2022 with the 95 % interval, how accurate it was in the backtest compared with the naive forecast, and what could make the forecast wrong.

The cell below refits the ETS model on all data up to 2021 and produces the 2022 forecast.

In [ ]:
final = fit_ets(y)
index_2022 = pd.date_range("2022-01-01", periods=12, freq="MS")
forecast_2022 = np.exp(final.get_prediction(start=index_2022[0], end=index_2022[-1]).summary_frame(alpha=0.05))
forecast_2022.round(0)

*Your recommendation:* ...

# Epilogue: what happened in 2022?

The test file contains the reviews of 2022 and 2023 (without ratings). Test reviews whose text also appeared in the training data were removed, and the data collection ended in 2023, so the counts are slightly lower than a full count would be. Treat this as an informal check, not as part of the model selection.

In [ ]:
later = pd.read_parquet(DATA / "test.parquet", columns=["date"])
actual_2022 = later.set_index("date").resample("MS").size()["2022"]
check = forecast_2022.assign(actual=actual_2022.to_numpy())
check["inside"] = (check["actual"] >= check["pi_lower"]) & (check["actual"] <= check["pi_upper"])
print(f"MAE 2022: {mae(check['actual'], check['mean']):.0f}; {check['inside'].sum()} of 12 months inside the interval")
ax = y["2019":].plot(figsize=(10, 3.5), color="black", label="training data")
check["actual"].plot(ax=ax, color="grey", label="2022 (test file)")
check["mean"].plot(ax=ax, ls="--", label="ETS forecast")
ax.fill_between(check.index, check["pi_lower"], check["pi_upper"], alpha=0.2, label="95 % interval")
ax.legend(loc="upper left")
check.round(0)

## Exercises

1. Add a sliding-window variant of the backtest (`TimeSeriesSplit(..., max_train_size=48)`). Do the rankings change?
2. Replace the target of the lag model by the year-on-year growth `log_y - log_y.shift(12)` and forecast `lag12 × exp(predicted growth)`. Does it extrapolate better in 2020 and 2021?
3. Optional: repeat the backtest with StatsForecast (`AutoETS`, `SeasonalNaive`, `cross_validation`), as in workbook 08, and compare the results.

In [ ]:
# Exercise 1: your code here
